In [1]:
from pathlib import Path
from kedro.framework.session import KedroSession
from kedro.framework.startup import bootstrap_project
import pandas as pd
import os
import uuid
import shutil

[12/02/25 11:52:42] INFO     Using                                                                  ]8;id=136658;file:///Users/bertrandgallice/code/Theia-Finance-Labs/crispy-kedro/.venv/lib/python3.10/site-packages/kedro/framework/project/__init__.py\__init__.py]8;;\:]8;id=660070;file:///Users/bertrandgallice/code/Theia-Finance-Labs/crispy-kedro/.venv/lib/python3.10/site-packages/kedro/framework/project/__init__.py#270\270]8;;\
                             '/Users/bertrandgallice/code/Theia-Finance-Labs/crispy-kedro/.venv/lib                
                             /python3.10/site-packages/kedro/framework/project/rich_logging.yml' as                
                             logging configuration.                                                                

In [2]:
os.environ["KEDRO_PACKAGE_NAME"] = "crispy_kedro"

workspace_dir = Path("workspace/results_V19")

tags=[
    "altrisk",
    # "reporting"
    ]


In [3]:
# Since the notebook is in ./notebooks, set the project path to the parent directory
current_dir = Path.cwd()
if current_dir.name == "notebooks":
    os.chdir(current_dir.parent)
    print(f"Changed directory from {current_dir} to {Path.cwd()}")
else:
    print(f"Already in correct directory: {current_dir}")

metadata = bootstrap_project(project_path=Path.cwd())

Changed directory from /Users/bertrandgallice/code/Theia-Finance-Labs/crispy-kedro/notebooks to /Users/bertrandgallice/code/Theia-Finance-Labs/crispy-kedro


[12/02/25 11:52:44] WARNING  /Users/bertrandgallice/code/Theia-Finance-Labs/crispy-kedro/.venv/lib/ ]8;id=804118;file:///usr/local/Cellar/python@3.10/3.10.16/Frameworks/Python.framework/Versions/3.10/lib/python3.10/warnings.py\warnings.py]8;;\:]8;id=816240;file:///usr/local/Cellar/python@3.10/3.10.16/Frameworks/Python.framework/Versions/3.10/lib/python3.10/warnings.py#109\109]8;;\
                             python3.10/site-packages/kedro/io/data_catalog.py:165:                                
                             KedroDeprecationWarning: `DataCatalog` has been deprecated and will be                
                             replaced by `KedroDataCatalog`, in Kedro 1.0.0.Currently some                         
                             `KedroDataCatalog` APIs have been retained for compatibility with                     
                             `DataCatalog`, including the `datasets` property and the                              
                             `get_datasets`, `_get_datasets`, `add`,` list`, `add_feed_dict`, and                  
                             `shallow_copy` methods. These will be removed or replaced with updated                
                             alternatives in Kedro 1.0.0. For more details, refer to the                           
                             documentation:                                                                        
                             https://docs.kedro.org/en/stable/data/index.html#kedrodatacatalog-expe                
                             rimental-feature                                                                      
                               warnings.warn(                                                                      
                                                                                                                   

In [4]:
assets = pd.read_csv("data/05_model_input/downloaded_assets.csv")
companies = pd.read_csv("data/05_model_input/downloaded_companies.csv")

assets_companies = pd.merge(
    assets, 
    companies, 
    on=["asset_id", "sector", "technology", "production_year"],
    how="left"
)

[12/02/25 11:52:47] WARNING  /var/folders/df/zghzv05d7xb8t9xy7y5ld_h40000gn/T/ipykernel_85062/35376 ]8;id=948004;file:///usr/local/Cellar/python@3.10/3.10.16/Frameworks/Python.framework/Versions/3.10/lib/python3.10/warnings.py\warnings.py]8;;\:]8;id=148651;file:///usr/local/Cellar/python@3.10/3.10.16/Frameworks/Python.framework/Versions/3.10/lib/python3.10/warnings.py#109\109]8;;\
                             70851.py:1: DtypeWarning: Columns (12) have mixed types. Specify dtype                
                             option on import or set low_memory=False.                                             
                               assets = pd.read_csv("data/05_model_input/downloaded_assets.csv")                   
                                                                                                                   

In [5]:
# Prepare companies summary to select a subset of companies later down

assets_companies_filtered = assets_companies[assets_companies["ownership_type"] == "direct"]
assets_companies_filtered = assets_companies_filtered[assets_companies_filtered["production_year"] == 2025]
assets_companies_filtered["capacity_owned"] = assets_companies_filtered["capacity"] * assets_companies_filtered["ownership_percentage"]
companies_summary = assets_companies_filtered.groupby(
    ["company_id", "company_name", "sector"], as_index=False
).agg(
    capacity_owned=("capacity_owned", "sum"),
    n_assets=("asset_id", "nunique"),
    n_countries=("country_iso2", "nunique")
).assign(
    capacity_owned=lambda x: x["capacity_owned"].astype(int),
)
companies_summary=  companies_summary.sort_values(by=["n_assets", "capacity_owned"], ascending=False).reset_index(drop=True)
companies_summary

,company_id,company_name,sector,capacity_owned,n_assets,n_countries
0,CP_7876088876044165226,other,Power,11464,358,31
1,CN_6660639238798673502,ENGIE SA,Power,17455,282,28
2,CN_5719632086864744401,EDF Renewables,Power,26700,279,26
3,CN_6785681074630732492,NextEra Energy Inc,Power,49174,256,3
4,CP_2362082173041183890,Iberdrola Renovables Energia SA,Power,8273,204,7
...,...,...,...,...,...,...
60536,CP_98321372751381949,Bem Querer hydroelectric plant,Power,0,1,1
60537,CP_989938990465740664,Shanxi Yangcheng Huangcheng Xiangfu Group Shis...,Coal,0,1,1
60538,CP_993937845921502505,Shanxi Qinyuan Guodao Jinyang Coal Industry Co...,Coal,0,1,1
60539,CP_996377999715439028,Balayan Bay Wind Power Project,Power,0,1,1


In [6]:

workspace_dir.mkdir(parents=True, exist_ok=True)
print(f"Created workspace directory: {workspace_dir}")


Created workspace directory: workspace/results_V19


In [7]:
 
# import logging

# # quiet down Kedro loggers
# for name in [
#     "kedro",
#     "kedro.framework",
#     "kedro.runner",
#     "kedro.io",
#     "kedro.pipeline",
#     "kedro.extras",
# ]:
#     logging.getLogger(name).setLevel(logging.WARNING)

# # (optional) quiet root logger too
# logging.getLogger().setLevel(logging.WARNING)

In [8]:
# Technology-based company sampling
import numpy as np

# Define sampling percentages for each technology (0.0 to 1.0)
tech_sampling_config = {
    'SolarCap - PV': 0.2,        # 10% sample
    'SolarCap - CSP': 1,       # 10% sample  
    'WindCap - Onshore': 0.2,    # 10% sample
    'WindCap - Offshore': 1,   # 10% sample
    'GasCap': 1,              # 5% sample
    'OilCap': 1,              # 5% sample
    'CoalCap': 1,             # 5% sample
    'GeothermalCap': 1,        # 20% sample
    'BF-EAF': 1,               # 10% sample
    'BF-BOF': 1,               # 10% sample
    'EAF': 1,                  # 10% sample
    'DRI-BOF': 1,              # 10% sample
    'DRI-EAF': 1,              # 10% sample
    'Oil': 1,                 # 5% sample
    'Gas': 1,                 # 5% sample
    'Both': 1,                 # 10% sample
    'BiomassCap': 1,           # 20% sample
    'NuclearCap': 1,           # 30% sample
    'HydroCap': 1,             # 10% sample
    'Coal': 1,                # 5% sample
    'Unknown': 1              # 5% sample
}

def sample_companies_by_technology(assets_companies_df, companies_summary_df, sampling_config, random_seed=42):
    """
    Sample companies based on technology percentages.
    
    Parameters:
    - assets_companies_df: DataFrame with asset-company relationships including technology
    - companies_summary_df: DataFrame with company summaries
    - sampling_config: Dictionary with technology -> sampling percentage
    - random_seed: Random seed for reproducibility
    
    Returns:
    - List of sampled company_ids
    """
    np.random.seed(random_seed)
    
    # Get companies by technology from assets data
    # Filter for 2025 and direct ownership like in companies_summary creation
    filtered_assets = assets_companies_df[
        (assets_companies_df["ownership_type"] == "direct") & 
        (assets_companies_df["production_year"] == 2025)
    ]
    
    # Group companies by technology
    companies_by_tech = filtered_assets.groupby('technology')['company_id'].unique().to_dict()
    
    sampled_company_ids = set()
    sampling_stats = {}
    
    for tech, companies_list in companies_by_tech.items():
        if tech in sampling_config:
            sample_pct = sampling_config[tech]
            n_companies = len(companies_list)
            n_sample = max(1, int(n_companies * sample_pct))  # At least 1 company if any exist
            
            # Sample companies for this technology
            sampled_tech_companies = np.random.choice(
                companies_list, 
                size=min(n_sample, n_companies), 
                replace=False
            )
            
            sampled_company_ids.update(sampled_tech_companies)
            sampling_stats[tech] = {
                'total_companies': n_companies,
                'sampled_companies': len(sampled_tech_companies),
                'sample_percentage': len(sampled_tech_companies) / n_companies if n_companies > 0 else 0
            }
        else:
            # If technology not in config, log it
            sampling_stats[tech] = {
                'total_companies': len(companies_list),
                'sampled_companies': 0,
                'sample_percentage': 0,
                'note': 'Technology not in sampling config'
            }
    
    # Convert to list and ensure companies exist in companies_summary
    final_company_ids = list(sampled_company_ids.intersection(set(companies_summary_df['company_id'])))
    
    # Print sampling statistics
    print("Technology Sampling Statistics:")
    print("=" * 50)
    for tech, stats in sampling_stats.items():
        if stats['sampled_companies'] > 0:
            print(f"{tech}: {stats['sampled_companies']}/{stats['total_companies']} companies ({stats['sample_percentage']:.1%})")
    
    print(f"\nTotal unique companies sampled: {len(final_company_ids)}")
    
    return final_company_ids

# Apply technology-based sampling
sampled_companies_by_tech = sample_companies_by_technology(
    assets_companies, 
    companies_summary, 
    tech_sampling_config
)

print(f"\nSample of selected company IDs: {sampled_companies_by_tech[:5]}...")


Technology Sampling Statistics:
BiomassCap: 1347/1347 companies (100.0%)
Both: 93/93 companies (100.0%)
Coal: 2078/2078 companies (100.0%)
CoalCap: 2764/2764 companies (100.0%)
Gas: 1782/1782 companies (100.0%)
GasCap: 2772/2772 companies (100.0%)
GeothermalCap: 163/163 companies (100.0%)
HydroCap: 2034/2034 companies (100.0%)
NuclearCap: 227/227 companies (100.0%)
Oil: 1979/1979 companies (100.0%)
OilCap: 1104/1104 companies (100.0%)
SolarCap - CSP: 170/170 companies (100.0%)
SolarCap - PV: 7283/36416 companies (20.0%)
Unknown: 355/355 companies (100.0%)
WindCap - Offshore: 934/934 companies (100.0%)
WindCap - Onshore: 2221/11108 companies (20.0%)

Total unique companies sampled: 23950

Sample of selected company IDs: ['CN_7411019103460932254', 'CN_7217524744994271816', 'CP_3185224859939516499', 'CP_6402210217893090120', 'CN_5498809099155088766']...


In [9]:
# companies_selection = ( pd.read_csv(
#     "workspace/results_v4/COFFEE_C3__company_granularity/asset_npv.csv")["company_id"]
#     .unique()
#     .tolist())


# Use technology-based sampling
# companies_selection = sampled_companies_by_tech

# Alternative: Use all companies (uncomment to disable sampling)
companies_selection = None  # all companies


# companies_selection = companies_summary.query(
#     "(n_assets < 100) & (n_assets > 5) & (sector == 'Power')"
#     ).company_id.tolist()
# len(companies_selection)

# to create an output to C/P in the config files to debug
# items = companies_selection
# yaml_list = "\n".join(f"- {item}" for item in items)
# print(yaml_list)

# companies_selection = [
#     # multinational megacorps
#     # "CP_7876088876044165226",
#     # "CN_9186444779649860568",
#     "CN_8600108312240451561",
#     "CP_1512176126791706747",
#     # big greentech owners
#     "CN_6660639238798673502",
#     "CN_6660639238798673502",
#     "CN_5719632086864744401",
#     # big carbontech owners
#     "CN_8600108312240451561",
#     "CN_7548398708980274705",
#     "CN_5252218731344992786",
#     # random other owners, with 10-20 assets
#     "CN_8249155112555313068",
#     "CP_7671368023139165011",
#     "CN_7263620466430749129",
#     "CP_5133603177600074280",
#     "CP_1405113695717703383",
#     "CN_7237425157254272056",
#     # random other owners, with <10 assets
#     "CN_1325960156574879189",
#     "CN_8258543408338880789",
#     "CN_4206272115616897750",
#     "CN_413233497131578182",
#     "CP_2845696436723078206",
#     "CN_6870637186458717950",
#     "CN_1465642096900403277",
#     "CN_4903484062166566625",
#     "CP_4899418540238054262",
#     "CP_1564781859061095794",
# ]


In [10]:
# import yaml
# with open("companies_selection.yml", "w") as f:
#     yaml.dump(companies_selection, f, default_flow_style=False)

# RUN

In [ ]:
# Define your parameter overrides

runs_configuration = {


    "AIM-CGE 2.2__asset_granularity_with_shock_staggered_shock_and_retirement":{  
        "company_ids": companies_selection,
        "reduce_granularity_from_asset_to_company_level": False,
        "apply_retirement_baseline": False,  # Apply retirement to baseline trajectories
        "apply_retirement_shock": True,     # Apply retirement to shock trajectories
        "apply_decreasing_staggered_shock":True,
        "baseline_scenario": "AR6_AIM/CGE 2.2_EN_NPi2020_1200f",
        "target_scenario": "AR6_AIM/CGE 2.2_EN_NPi2020_900f",
        "use_frozen_capacity_for_fixed_costs": False,
        "max_forecast_horizon":5
    },
    # "AIM-CGE 2.2__asset_granularity_with_staggered_shock_and_retirement":{  
    #     "company_ids": companies_selection,
    #     "reduce_granularity_from_asset_to_company_level": False,
    #     "apply_retirement_baseline": True,  # Apply retirement to baseline trajectories
    #     "apply_retirement_shock": True,     # Apply retirement to shock trajectories
    #     "apply_decreasing_staggered_shock":True,
    #     "baseline_scenario": "AR6_AIM/CGE 2.2_EN_NPi2020_1200f",
    #     "target_scenario": "AR6_AIM/CGE 2.2_EN_NPi2020_900f",
    #     "use_frozen_capacity_for_fixed_costs": False,
    #     "max_forecast_horizon":5
    # },
    # "AIM-CGE 2.2__company_granularity_with_continued_omcost":{
    #     "company_ids": companies_selection,
    #     "reduce_granularity_from_asset_to_company_level": True,
    #     "apply_retirement_baseline": False,  # Apply retirement to baseline trajectories
    #     "apply_retirement_shock": False,     # Apply retirement to shock trajectories
    #     "apply_decreasing_staggered_shock":False,
    #     "baseline_scenario": "AR6_AIM/CGE 2.2_EN_NPi2020_1200f",
    #     "target_scenario": "AR6_AIM/CGE 2.2_EN_NPi2020_900f" ,
    #     "use_frozen_capacity_for_fixed_costs": True,
    #     "max_forecast_horizon":1
    # },
    # "AIM-CGE 2.2__asset_granularity_with_continued_omcost_and_retirement":{  
    #     "company_ids": companies_selection,
    #     "reduce_granularity_from_asset_to_company_level": False,
    #     "apply_retirement":True,
    #     "apply_decreasing_staggered_shock":False,
    #     "baseline_scenario": "AR6_AIM/CGE 2.2_EN_NPi2020_1200f",
    #     "target_scenario": "AR6_AIM/CGE 2.2_EN_NPi2020_900f",
    #     "use_frozen_capacity_for_fixed_costs": True,
    # },    
    # "AIM-CGE 2.2__asset_granularity_with_continued_omcost_and_staggered_shock_and_retirement":{  
    #     "company_ids": companies_selection,
    #     "reduce_granularity_from_asset_to_company_level": False,
    #     "apply_retirement":True,
    #     "apply_decreasing_staggered_shock":True,
    #     "baseline_scenario": "AR6_AIM/CGE 2.2_EN_NPi2020_1200f",
    #     "target_scenario": "AR6_AIM/CGE 2.2_EN_NPi2020_900f",
    #     "use_frozen_capacity_for_fixed_costs": True,
    # },

}

# runs_configuration = {

#     "NGFS_GCAM_B2DS__company_granularity":{
#         "company_ids": companies_selection,
#         "reduce_granularity_from_asset_to_company_level": True,
#         "apply_retirement":False,
#         "apply_decreasing_staggered_shock":False,
#         "baseline_scenario": "AR6_GCAM 6.0 NGFS_Current Policies",
#         "target_scenario": "AR6_GCAM 6.0 NGFS_Below 2°C" 
        
#     },
#     "NGFS_GCAM_B2DS__asset_granularity_with_staggered_shock_and_retirement":{  
#         "company_ids": companies_selection,
#         "reduce_granularity_from_asset_to_company_level": False,
#         "apply_retirement":True,
#         "apply_decreasing_staggered_shock":True,
#         "baseline_scenario": "AR6_GCAM 6.0 NGFS_Current Policies",
#         "target_scenario": "AR6_GCAM 6.0 NGFS_Below 2°C" 
#     },

#     "NGFS_GCAM_NZ2050__company_granularity":{
#         "company_ids": companies_selection,
#         "reduce_granularity_from_asset_to_company_level": True,
#         "apply_retirement":False,
#         "apply_decreasing_staggered_shock":False,
#         "baseline_scenario": "AR6_GCAM 6.0 NGFS_Current Policies",
#         "target_scenario": "AR6_GCAM 6.0 NGFS_Net Zero 2050" 
        
#     },
#     "NGFS_GCAM_NZ2050__asset_granularity_with_staggered_shock_and_retirement":{  
#         "company_ids": companies_selection,
#         "reduce_granularity_from_asset_to_company_level": False,
#         "apply_retirement":True,
#         "apply_decreasing_staggered_shock":True,
#         "baseline_scenario": "AR6_GCAM 6.0 NGFS_Current Policies",
#         "target_scenario": "AR6_GCAM 6.0 NGFS_Net Zero 2050" 
#     },




#     "NGFS_MESSAGE_B2DS__company_granularity":{
#         "company_ids": companies_selection,
#         "reduce_granularity_from_asset_to_company_level": True,
#         "apply_retirement":False,
#         "apply_decreasing_staggered_shock":False,
#         "baseline_scenario": "AR6_MESSAGEix-GLOBIOM 2.0-M-R12-NGFS_Current Policies",
#         "target_scenario": "AR6_MESSAGEix-GLOBIOM 2.0-M-R12-NGFS_Below 2°C" 
        
#     },
#     "NGFS_MESSAGE_B2DS__asset_granularity_with_staggered_shock_and_retirement":{  
#         "company_ids": companies_selection,
#         "reduce_granularity_from_asset_to_company_level": False,
#         "apply_retirement":True,
#         "apply_decreasing_staggered_shock":True,
#         "baseline_scenario": "AR6_MESSAGEix-GLOBIOM 2.0-M-R12-NGFS_Current Policies",
#         "target_scenario": "AR6_MESSAGEix-GLOBIOM 2.0-M-R12-NGFS_Below 2°C" 
#     },

#     "NGFS_MESSAGE_NZ2050__company_granularity":{
#         "company_ids": companies_selection,
#         "reduce_granularity_from_asset_to_company_level": True,
#         "apply_retirement":False,
#         "apply_decreasing_staggered_shock":False,
#         "baseline_scenario": "AR6_MESSAGEix-GLOBIOM 2.0-M-R12-NGFS_Current Policies",
#         "target_scenario": "AR6_MESSAGEix-GLOBIOM 2.0-M-R12-NGFS_Net Zero 2050" 
        
#     },
#     "NGFS_MESSAGE_NZ2050__asset_granularity_with_staggered_shock_and_retirement":{  
#         "company_ids": companies_selection,
#         "reduce_granularity_from_asset_to_company_level": False,
#         "apply_retirement":True,
#         "apply_decreasing_staggered_shock":True,
#         "baseline_scenario": "AR6_MESSAGEix-GLOBIOM 2.0-M-R12-NGFS_Current Policies",
#         "target_scenario": "AR6_MESSAGEix-GLOBIOM 2.0-M-R12-NGFS_Net Zero 2050" 
#     },




#     "NGFS_REMIND_B2DS__company_granularity":{
#         "company_ids": companies_selection,
#         "reduce_granularity_from_asset_to_company_level": True,
#         "apply_retirement":False,
#         "apply_decreasing_staggered_shock":False,
#         "baseline_scenario": "AR6_REMIND-MAgPIE 3.3-4.8_Current Policies",
#         "target_scenario": "AR6_REMIND-MAgPIE 3.3-4.8_Below 2°C" 
        
#     },
#     "NGFS_REMIND_B2DS__asset_granularity_with_staggered_shock_and_retirement":{  
#         "company_ids": companies_selection,
#         "reduce_granularity_from_asset_to_company_level": False,
#         "apply_retirement":True,
#         "apply_decreasing_staggered_shock":True,
#         "baseline_scenario": "AR6_REMIND-MAgPIE 3.3-4.8_Current Policies",
#         "target_scenario": "AR6_REMIND-MAgPIE 3.3-4.8_Below 2°C" 
#     },

#     "NGFS_REMIND_NZ2050__company_granularity":{
#         "company_ids": companies_selection,
#         "reduce_granularity_from_asset_to_company_level": True,
#         "apply_retirement":False,
#         "apply_decreasing_staggered_shock":False,
#         "baseline_scenario": "AR6_REMIND-MAgPIE 3.3-4.8_Current Policies",
#         "target_scenario": "AR6_REMIND-MAgPIE 3.3-4.8_Net Zero 2050" 
        
#     },
#     "NGFS_REMIND_NZ2050__asset_granularity_with_staggered_shock_and_retirement":{  
#         "company_ids": companies_selection,
#         "reduce_granularity_from_asset_to_company_level": False,
#         "apply_retirement":True,
#         "apply_decreasing_staggered_shock":True,
#         "baseline_scenario": "AR6_REMIND-MAgPIE 3.3-4.8_Current Policies",
#         "target_scenario": "AR6_REMIND-MAgPIE 3.3-4.8_Net Zero 2050" 
#     },


# }

# runs_configuration = {


    # # "IMACLIM_ADVANCE__company_granularity":{
    # #     "company_ids": companies_selection,
    # #     "reduce_granularity_from_asset_to_company_level": True,
    # #     "apply_retirement":False,
    # #     "apply_decreasing_staggered_shock":False,
    # #     "baseline_scenario": "AR6_IMACLIM 1.1_ADVANCE_NoPolicy_WP6",
    # #     "target_scenario": "AR6_IMACLIM 1.1_ADVANCE_INDC_WP6" 
        
    # # },
    # # "IMACLIM_ADVANCE__asset_granularity_with_staggered_shock_and_retirement":{  
    # #     "company_ids": companies_selection,
    # #     "reduce_granularity_from_asset_to_company_level": False,
    # #     "apply_retirement":True,
    # #     "apply_decreasing_staggered_shock":True,
    # #     "baseline_scenario": "AR6_IMACLIM 1.1_ADVANCE_NoPolicy_WP6",
    # #     "target_scenario": "AR6_IMACLIM 1.1_ADVANCE_INDC_WP6" 
    # # },



    # # "IMACLIM_SSP3__company_granularity":{
    # #     "company_ids": companies_selection,
    # #     "reduce_granularity_from_asset_to_company_level": True,
    # #     "apply_retirement":False,
    # #     "apply_decreasing_staggered_shock":False,
    # #     "baseline_scenario": "AR6_IMACLIM 1.1_SSP3_NoPolicy_TranspBase",
    # #     "target_scenario": "AR6_IMACLIM 1.1_SSP3_RCP45_TranspBase" 
        
    # # },
    # # "IMACLIM_SSP3__asset_granularity_with_staggered_shock_and_retirement":{  
    # #     "company_ids": companies_selection,
    # #     "reduce_granularity_from_asset_to_company_level": False,
    # #     "apply_retirement":True,
    # #     "apply_decreasing_staggered_shock":True,
    # #     "baseline_scenario": "AR6_IMACLIM 1.1_SSP3_NoPolicy_TranspBase",
    # #     "target_scenario": "AR6_IMACLIM 1.1_SSP3_RCP45_TranspBase" 
    # # },





    # "COFFEE_1000___company_granularity":{
    #     "company_ids": companies_selection,
    #     "reduce_granularity_from_asset_to_company_level": True,
    #     "apply_retirement":False,
    #     "apply_decreasing_staggered_shock":False,
    #     "baseline_scenario": "AR6_COFFEE 1.1_EN_NPi2020_1000_COV",
    #     "target_scenario": "AR6_COFFEE 1.1_EN_INDCi2030_1000_COV_NDCp" 
        
    # },
    # "COFFEE_1000__asset_granularity_with_staggered_shock_and_retirement":{  
    #     "company_ids": companies_selection,
    #     "reduce_granularity_from_asset_to_company_level": False,
    #     "apply_retirement":True,
    #     "apply_decreasing_staggered_shock":True,
    #     "baseline_scenario": "AR6_COFFEE 1.1_EN_NPi2020_1000_COV",
    #     "target_scenario": "AR6_COFFEE 1.1_EN_INDCi2030_1000_COV_NDCp" 
    # },


    # "COFFEE_600___company_granularity":{
    #     "company_ids": companies_selection,
    #     "reduce_granularity_from_asset_to_company_level": True,
    #     "apply_retirement":False,
    #     "apply_decreasing_staggered_shock":False,
    #     "baseline_scenario": "AR6_COFFEE 1.1_EN_NPi2020_600_COV",
    #     "target_scenario": "AR6_COFFEE 1.1_EN_INDCi2030_600_COV_NDCp" 
        
    # },
    # "COFFEE_600__asset_granularity_with_staggered_shock_and_retirement":{  
    #     "company_ids": companies_selection,
    #     "reduce_granularity_from_asset_to_company_level": False,
    #     "apply_retirement":True,
    #     "apply_decreasing_staggered_shock":True,
    #     "baseline_scenario": "AR6_COFFEE 1.1_EN_NPi2020_600_COV",
    #     "target_scenario": "AR6_COFFEE 1.1_EN_INDCi2030_600_COV_NDCp" 
    # },


    # # "THIAM-GRANTHAM_2DEG2030___company_granularity":{
    # #     "company_ids": companies_selection,
    # #     "reduce_granularity_from_asset_to_company_level": True,
    # #     "apply_retirement":False,
    # #     "apply_decreasing_staggered_shock":False,
    # #     "baseline_scenario": "AR6_TIAM-Grantham 3.2_CO_CurPol",
    # #     "target_scenario": "AR6_TIAM-Grantham 3.2_CO_2Deg2030" 
        
    # # },
    # "THIAM-GRANTHAM_2DEG2030__asset_granularity_with_staggered_shock_and_retirement":{  
    #     "company_ids": companies_selection,
    #     "reduce_granularity_from_asset_to_company_level": False,
    #     "apply_retirement":True,
    #     "apply_decreasing_staggered_shock":True,
    #     "baseline_scenario": "AR6_TIAM-Grantham 3.2_CO_CurPol",
    #     "target_scenario": "AR6_TIAM-Grantham 3.2_CO_2Deg2030" 
    # },


    # "THIAM-GRANTHAM_BRIDGE___company_granularity":{
    #     "company_ids": companies_selection,
    #     "reduce_granularity_from_asset_to_company_level": True,
    #     "apply_retirement":False,
    #     "apply_decreasing_staggered_shock":False,
    #     "baseline_scenario": "AR6_TIAM-Grantham 3.2_CO_CurPol",
    #     "target_scenario": "AR6_TIAM-Grantham 3.2_CO_Bridge" 
        
    # },
    # "THIAM-GRANTHAM_BRIDGE__asset_granularity_with_staggered_shock_and_retirement":{  
    #     "company_ids": companies_selection,
    #     "reduce_granularity_from_asset_to_company_level": False,
    #     "apply_retirement":True,
    #     "apply_decreasing_staggered_shock":True,
    #     "baseline_scenario": "AR6_TIAM-Grantham 3.2_CO_CurPol",
    #     "target_scenario": "AR6_TIAM-Grantham 3.2_CO_Bridge" 
    # },




# }



# runs_configuration = {
#     # "AIM-CGE 2.2__company_granularity":{
#     #     "company_ids": companies_selection,
#     #     "reduce_granularity_from_asset_to_company_level": True,
#     #     "apply_retirement":False,
#     #     "apply_decreasing_staggered_shock":False,
#     #     "baseline_scenario": "AR6_AIM/CGE 2.2_EN_NPi2020_1200f",
#     #     "target_scenario": "AR6_AIM/CGE 2.2_EN_NPi2020_900f" 
#     # },
#     # "AIM-CGE 2.2__asset_granularity_with_staggered_shock_and_retirement":{  
#     #     "company_ids": companies_selection,
#     #     "reduce_granularity_from_asset_to_company_level": False,
#     #     "apply_retirement":True,
#     #     "apply_decreasing_staggered_shock":True,
#     #     "baseline_scenario": "AR6_AIM/CGE 2.2_EN_NPi2020_1200f",
#     #     "target_scenario": "AR6_AIM/CGE 2.2_EN_NPi2020_900f" 
#     # },

#     # "COFFEE_1-1__company_granularity":{
#     #     "company_ids": companies_selection,
#     #     "reduce_granularity_from_asset_to_company_level": True,
#     #     "apply_retirement":False,
#     #     "apply_decreasing_staggered_shock":False,
#     #     "baseline_scenario": "AR6_COFFEE 1.1_CO_CurPol",
#     #     "target_scenario": "AR6_COFFEE 1.1_EN_NPi2020_400f_lowBECCS" 
        
#     # },
#     # "COFFEE_1-1__asset_granularity_with_staggered_shock_and_retirement":{  
#     #     "company_ids": companies_selection,
#     #     "reduce_granularity_from_asset_to_company_level": False,
#     #     "apply_retirement":True,
#     #     "apply_decreasing_staggered_shock":True,
#     #     "baseline_scenario": "AR6_COFFEE 1.1_CO_CurPol",
#     #     "target_scenario": "AR6_COFFEE 1.1_EN_NPi2020_400f_lowBECCS" 
#     # },




#     # "REMIND-MAgPIE_DeepElec__company_granularity":{
#     #     "company_ids": companies_selection,
#     #     "reduce_granularity_from_asset_to_company_level": True,
#     #     "apply_retirement":False,
#     #     "apply_decreasing_staggered_shock":False,
#     #     "baseline_scenario": "AR6_REMIND-MAgPIE 2.1-4.3_DeepElec_SSP2_Base",
#     #     "target_scenario": "AR6_REMIND-MAgPIE 2.1-4.3_DeepElec_SSP2_ HighRE_Budg900" 
#     # },
    
#     # "REMIND-MAgPIE_DeepElec__asset_granularity_with_staggered_shock_and_retirement":{  
#     #     "company_ids": companies_selection,
#     #     "reduce_granularity_from_asset_to_company_level": False,
#     #     "apply_retirement":True,
#     #     "apply_decreasing_staggered_shock":True,
#     #     "baseline_scenario": "AR6_REMIND-MAgPIE 2.1-4.3_DeepElec_SSP2_Base",
#     #     "target_scenario": "AR6_REMIND-MAgPIE 2.1-4.3_DeepElec_SSP2_ HighRE_Budg900" 
#     # },


#     # "MESSAGEix__company_granularity":{
#     #     "company_ids": companies_selection,
#     #     "reduce_granularity_from_asset_to_company_level": True,
#     #     "apply_retirement":False,
#     #     "apply_decreasing_staggered_shock":False,
#     #     "baseline_scenario": "AR6_MESSAGEix-GLOBIOM_GEI 1.0_SSP2_int_mc_50",
#     #     "target_scenario": "AR6_MESSAGEix-GLOBIOM_GEI 1.0_SSP2_openres_lc_50" 
        
#     # },
#     # "MESSAGEix__asset_granularity_with_staggered_shock_and_retirement":{  
#     #     "company_ids": companies_selection,
#     #     "reduce_granularity_from_asset_to_company_level": False,
#     #     "apply_retirement":True,
#     #     "apply_decreasing_staggered_shock":True,
#     #     "baseline_scenario": "AR6_MESSAGEix-GLOBIOM_GEI 1.0_SSP2_int_mc_50",
#     #     "target_scenario": "AR6_MESSAGEix-GLOBIOM_GEI 1.0_SSP2_openres_lc_50" 
#     # },



#     # "REMIND-MAgPIE_Susdev__company_granularity":{
#     #     "company_ids": companies_selection,
#     #     "reduce_granularity_from_asset_to_company_level": True,
#     #     "apply_retirement":False,
#     #     "apply_decreasing_staggered_shock":False,
#     #     "baseline_scenario": "AR6_REMIND-MAgPIE 2.1-4.2_SusDev_SDP-NPi",
#     #     "target_scenario": "AR6_REMIND-MAgPIE 2.1-4.2_SusDev_SDP-PkBudg1000" 
#     # },
    
#     # "REMIND-MAgPIE_Susdev__asset_granularity_with_staggered_shock_and_retirement":{  
#     #     "company_ids": companies_selection,
#     #     "reduce_granularity_from_asset_to_company_level": False,
#     #     "apply_retirement":True,
#     #     "apply_decreasing_staggered_shock":True,
#     #     "baseline_scenario": "AR6_REMIND-MAgPIE 2.1-4.2_SusDev_SDP-NPi",
#     #     "target_scenario": "AR6_REMIND-MAgPIE 2.1-4.2_SusDev_SDP-PkBudg1000" 
#     # },

#     # "WITCH_5-0__company_granularity":{
#     #     "company_ids": companies_selection,
#     #     "reduce_granularity_from_asset_to_company_level": True,
#     #     "apply_retirement":False,
#     #     "apply_decreasing_staggered_shock":False,
#     #     "baseline_scenario": "AR6_WITCH 5.0_CO_CurPol",
#     #     "target_scenario": "AR6_WITCH 5.0_CO_Bridge" 
        
#     # },

#     # "WITCH_5-0__asset_granularity_with_staggered_shock_and_retirement":{  
#     #     "company_ids": companies_selection,
#     #     "reduce_granularity_from_asset_to_company_level": False,
#     #     "apply_retirement":True,
#     #     "apply_decreasing_staggered_shock":True,
#     #     "baseline_scenario": "AR6_WITCH 5.0_CO_CurPol",
#     #     "target_scenario": "AR6_WITCH 5.0_CO_Bridge" 
#     # },

# }


# {
    # "COFFEE_C3__company_granularity":{
    #     "company_ids": companies_selection,
    #     "reduce_granularity_from_asset_to_company_level": True,
    #     "apply_retirement":False,
    #     "apply_decreasing_staggered_shock":False,
    #     "baseline_scenario": "AR6_COFFEE 1.1_CO_CurPol",
    #     "target_scenario": "AR6_COFFEE 1.1_EN_NPi2020_800" # C3
        
    # },
    # "COFFEE_C3__asset_granularity_with_staggered_shock_and_retirement":{  
    #     "company_ids": companies_selection,
    #     "reduce_granularity_from_asset_to_company_level": False,
    #     "apply_retirement":True,
    #     "apply_decreasing_staggered_shock":True,
    #     "baseline_scenario": "AR6_WITCH 5.0_CO_CurPol",
    #     "target_scenario": "AR6_COFFEE 1.1_EN_NPi2020_800" # C3
    # },

    # "COFFEE_C2__company_granularity":{
    #     "company_ids": companies_selection,
    #     "reduce_granularity_from_asset_to_company_level": True,
    #     "apply_retirement":False,
    #     "apply_decreasing_staggered_shock":False,
    #     "baseline_scenario": "AR6_COFFEE 1.1_CO_CurPol",
    #     "target_scenario": "AR6_COFFEE 1.1_EN_NPi2020_400f", # C2
        
    # },
    # "COFFEE_C2__asset_granularity_with_staggered_shock_and_retirement":{  
    #     "company_ids": companies_selection,
    #     "reduce_granularity_from_asset_to_company_level": False,
    #     "apply_retirement":True,
    #     "apply_decreasing_staggered_shock":True,
    #     "baseline_scenario": "AR6_COFFEE 1.1_CO_CurPol",
    #     "target_scenario": "AR6_COFFEE 1.1_EN_NPi2020_400f", # C2
    # },

    # "COFFEE_C5__company_granularity":{
    #     "company_ids": companies_selection,
    #     "reduce_granularity_from_asset_to_company_level": True,
    #     "apply_retirement":False,
    #     "apply_decreasing_staggered_shock":False,
    #     "baseline_scenario": "AR6_COFFEE 1.1_CO_CurPol",
    #     "target_scenario": "AR6_COFFEE 1.1_EN_NPi2020_1400", # C5
        
    # },
    # "COFFEE_C5__asset_granularity_with_staggered_shock_and_retirement":{  
    #     "company_ids": companies_selection,
    #     "reduce_granularity_from_asset_to_company_level": False,
    #     "apply_retirement":True,
    #     "apply_decreasing_staggered_shock":True,
    #     "baseline_scenario": "AR6_COFFEE 1.1_CO_CurPol",
    #     "target_scenario": "AR6_COFFEE 1.1_EN_NPi2020_1400", # C5
    # },
# }



In [12]:
from IPython.display import clear_output
import sys
import traceback
from io import StringIO

all_late_sudden_trajectories = {}
all_staggered_shock_results = {}
all_companies_npvs = {}
all_run_params = {}

total_runs = len(runs_configuration)

for idx, (run_name, run_params) in enumerate(runs_configuration.items(), start=1):
    clear_output(wait=True)  # clears the cell output each iteration
    
    print("================================================")
    print("================================================")
    print(f"Running {run_name}...")
    print(f"Run {idx}/{total_runs}")
    print("================================================")
    print("================================================")
    
    try:
        with KedroSession.create(
            project_path=Path.cwd(),
            extra_params=run_params,
        ) as session:
            session.run(pipeline_name="__default__", tags=tags)

            run_id = uuid.uuid4()

            # late_sudden_trajectories = session.load("late_sudden_trajectories")
            late_sudden_trajectories = pd.read_csv(
                "data/07_model_output/companies_late_sudden_trajectories.csv"
            )
            late_sudden_trajectories["run_id"] = run_id
            staggered_shock_results = pd.read_csv(
                "data/07_model_output/asset_level_staggered_shock.csv"
            )
            staggered_shock_results["run_id"] = run_id
            yearly_npv_trajectories = pd.read_csv(
                "data/07_model_output/yearly_npv_trajectories.csv"
            )
            yearly_npv_trajectories["run_id"] = run_id
            asset_npvs = pd.read_csv(
                "data/07_model_output/asset_npv.csv"
            )
            asset_npvs["run_id"] = run_id
            company_technology_npvs = pd.read_csv(
                "data/07_model_output/company_technology_npv.csv"
            )
            company_technology_npvs["run_id"] = run_id
            companies_npvs = pd.read_csv(
                "data/07_model_output/company_npv.csv"
            )
            companies_npvs["run_id"] = run_id

            run_params_df = pd.DataFrame([run_params])
            run_params_df["run_id"] = run_id

            all_late_sudden_trajectories[run_name] = late_sudden_trajectories
            all_staggered_shock_results[run_name] = staggered_shock_results
            all_companies_npvs[run_name] = companies_npvs
            all_run_params[run_name] = run_params_df

            # Copy plot folders to {workspace_dir}/{run_name}/
            run_workspace_dir = workspace_dir / run_name
            run_workspace_dir.mkdir(parents=True, exist_ok=True)

            late_sudden_trajectories.to_csv(run_workspace_dir / "all_late_sudden_trajectories.csv", index=False)
            staggered_shock_results.to_csv(run_workspace_dir / "asset_level_staggered_shock.csv", index=False)
            asset_npvs.to_csv(run_workspace_dir / "asset_npv.csv", index=False)
            company_technology_npvs.to_csv(run_workspace_dir / "company_technology_npv.csv", index=False)
            companies_npvs.to_csv(run_workspace_dir / "company_npv.csv", index=False)
            yearly_npv_trajectories.to_csv(run_workspace_dir / "yearly_npv_trajectories.csv", index=False)
            run_params_df.to_csv(run_workspace_dir / "run_params.csv", index=False)
            
            if "reporting" in tags:
                # Copy companies_trajectories_plots
                src_trajectories = Path("data/08_reporting/companies_trajectories_plots")
                dst_trajectories = run_workspace_dir / "companies_trajectories_plots"
                if src_trajectories.exists():
                    if dst_trajectories.exists():
                        shutil.rmtree(dst_trajectories)
                    shutil.copytree(src_trajectories, dst_trajectories)
                    print(f"Copied companies_trajectories_plots to {dst_trajectories}")
                
                # Copy companies_staggered_shock_plots  
                src_staggered = Path("data/08_reporting/companies_staggered_shock_plots")
                dst_staggered = run_workspace_dir / "companies_staggered_shock_plots"
                if src_staggered.exists():
                    if dst_staggered.exists():
                        shutil.rmtree(dst_staggered)
                    shutil.copytree(src_staggered, dst_staggered)
                    print(f"Copied companies_staggered_shock_plots to {dst_staggered}")

                # Copy companies_staggered_shock_plots  
                src_staggered = Path("data/08_reporting/asset_financial_trajectories")
                dst_staggered = run_workspace_dir / "asset_financial_trajectories"
                if src_staggered.exists():
                    if dst_staggered.exists():
                        shutil.rmtree(dst_staggered)
                    shutil.copytree(src_staggered, dst_staggered)
                    print(f"Copied asset_financial_trajectories to {dst_staggered}")
                
        print(f"✅ Successfully completed {run_name}")
        
    except Exception as e:
        # Capture the error and traceback
        error_msg = f"❌ Error in {run_name}:\n"
        error_msg += f"Exception: {str(e)}\n"
        error_msg += f"Traceback:\n{traceback.format_exc()}\n"
        
        # Save error to file in the same root as the run folder
        error_file = workspace_dir / f"{run_name}_error.txt"
        with open(error_file, 'w') as f:
            f.write(error_msg)
        
        print(f"❌ Error in {run_name} - saved to {error_file}")
        print(f"Continuing with next run...")
        
        # Continue to next iteration
        continue
    else:
        print(f"{'✅'*total_runs} Successfully completed all runs")


Running AIM-CGE 2.2__company_granularity_with_continued_omcost...
Run 2/2


Python(53483) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
Python(53484) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


[12/02/25 13:48:16] INFO     Kedro project crispy-kedro                                              ]8;id=68457;file:///Users/bertrandgallice/code/Theia-Finance-Labs/crispy-kedro/.venv/lib/python3.10/site-packages/kedro/framework/session/session.py\session.py]8;;\:]8;id=657322;file:///Users/bertrandgallice/code/Theia-Finance-Labs/crispy-kedro/.venv/lib/python3.10/site-packages/kedro/framework/session/session.py#329\329]8;;\

                    WARNING  /Users/bertrandgallice/code/Theia-Finance-Labs/crispy-kedro/.venv/lib/ ]8;id=341758;file:///usr/local/Cellar/python@3.10/3.10.16/Frameworks/Python.framework/Versions/3.10/lib/python3.10/warnings.py\warnings.py]8;;\:]8;id=550352;file:///usr/local/Cellar/python@3.10/3.10.16/Frameworks/Python.framework/Versions/3.10/lib/python3.10/warnings.py#109\109]8;;\
                             python3.10/site-packages/kedro/io/data_catalog.py:165:                                
                             KedroDeprecationWarning: `DataCatalog` has been deprecated and will be                
                             replaced by `KedroDataCatalog`, in Kedro 1.0.0.Currently some                         
                             `KedroDataCatalog` APIs have been retained for compatibility with                     
                             `DataCatalog`, including the `datasets` property and the                              
                             `get_datasets`, `_get_datasets`, `add`,` list`, `add_feed_dict`, and                  
                             `shallow_copy` methods. These will be removed or replaced with updated                
                             alternatives in Kedro 1.0.0. For more details, refer to the                           
                             documentation:                                                                        
                             https://docs.kedro.org/en/stable/data/index.html#kedrodatacatalog-expe                
                             rimental-feature                                                                      
                               warnings.warn(                                                                      
                                                                                                                   

[12/02/25 13:48:17] INFO     Using synchronous mode for loading and saving data. Use the    ]8;id=49095;file:///Users/bertrandgallice/code/Theia-Finance-Labs/crispy-kedro/.venv/lib/python3.10/site-packages/kedro/runner/sequential_runner.py\sequential_runner.py]8;;\:]8;id=791070;file:///Users/bertrandgallice/code/Theia-Finance-Labs/crispy-kedro/.venv/lib/python3.10/site-packages/kedro/runner/sequential_runner.py#68\68]8;;\
                             --async flag for potential performance gains.                                         
                             https://docs.kedro.org/en/stable/nodes_and_pipelines/run_a_pip                        
                             eline.html#load-and-save-asynchronously                                               

...

                    INFO     Completed 43 out of 43 tasks                                             ]8;id=308488;file:///Users/bertrandgallice/code/Theia-Finance-Labs/crispy-kedro/.venv/lib/python3.10/site-packages/kedro/runner/runner.py\runner.py]8;;\:]8;id=879239;file:///Users/bertrandgallice/code/Theia-Finance-Labs/crispy-kedro/.venv/lib/python3.10/site-packages/kedro/runner/runner.py#245\245]8;;\

                    INFO     Pipeline execution completed successfully.                               ]8;id=876077;file:///Users/bertrandgallice/code/Theia-Finance-Labs/crispy-kedro/.venv/lib/python3.10/site-packages/kedro/runner/runner.py\runner.py]8;;\:]8;id=566078;file:///Users/bertrandgallice/code/Theia-Finance-Labs/crispy-kedro/.venv/lib/python3.10/site-packages/kedro/runner/runner.py#131\131]8;;\

                    INFO     Loading data from companies_late_sudden_trajectories_corrected     ]8;id=395741;file:///Users/bertrandgallice/code/Theia-Finance-Labs/crispy-kedro/.venv/lib/python3.10/site-packages/kedro/io/data_catalog.py\data_catalog.py]8;;\:]8;id=47495;file:///Users/bertrandgallice/code/Theia-Finance-Labs/crispy-kedro/.venv/lib/python3.10/site-packages/kedro/io/data_catalog.py#401\401]8;;\
                             (MemoryDataset)...                                                                    

[12/02/25 16:26:29] INFO     Loading data from _temp_alignments_validated (MemoryDataset)...    ]8;id=726621;file:///Users/bertrandgallice/code/Theia-Finance-Labs/crispy-kedro/.venv/lib/python3.10/site-packages/kedro/io/data_catalog.py\data_catalog.py]8;;\:]8;id=383284;file:///Users/bertrandgallice/code/Theia-Finance-Labs/crispy-kedro/.venv/lib/python3.10/site-packages/kedro/io/data_catalog.py#401\401]8;;\

✅ Successfully completed AIM-CGE 2.2__company_granularity_with_continued_omcost
✅✅ Successfully completed all runs
